# Capstone build --- Chapter 11: Failure Modes and Design of Experiments

Chapter~10 scored one trajectory. A handful of trajectories does not establish that an agent is safe, because the cases that break it are the ones no one thought to write. Chapter~11 replaces hand-picked cases with a design: name the factors that make a complaint hard --- product, regulatory exposure, whether the message carries PII or an injection attempt --- and cover their combinations with a balanced set. It then names the failure modes a test must inject to exercise the agent's defenses rather than only its happy path.

## A balanced design over the hard factors

`balanced_design` builds a set of factor combinations that covers each level of each factor roughly equally, so the test suite is not accidentally skewed toward easy cases. The factors below are the dimensions along which a complaint case varies in ways that stress the agent.

In [ ]:
from agentlab.evaluation.doe import balanced_design, coverage_report

factors = {
    'product': ['checking_account', 'credit_card', 'mortgage'],
    'regulatory': ['none', 'UDAAP', 'reg_x'],
    'adversarial': ['clean', 'pii', 'prompt_injection'],
}
design = balanced_design(factors, num_cases=12, seed=0)
for i, case in enumerate(design):
    print(f'{i:2d}: {case}')
print('coverage:', coverage_report(design, factors))

## The failure modes a test injects

A test that only sends well-formed complaints never exercises the gates. The failure-mode catalog names the adversarial and degenerate behaviors a suite must include: a prompt-injection payload, a malformed call, a hallucinated citation, a wedged loop. Each is a factory returning a `FailureInjection` that can be applied to a scenario.

In [ ]:
from agentlab.evaluation.failure_modes import (
    prompt_injection, malformed_call, hallucinated_citation, infinite_loop,
)
for factory in (prompt_injection, malformed_call, hallucinated_citation, infinite_loop):
    inj = factory()
    print(f'{inj.mode!s:24s} {inj.description}')

## From design to test cases

`generate_test_cases` turns the design into `TestCase`s, each carrying a `TaskSpec`, the user message, the expected behavior and the factor combination it realizes. This is the suite Chapter~16 runs the whole agent against; here it shows the shape of a generated case.

In [ ]:
from agentlab.evaluation.test_cases import generate_test_cases

cases = generate_test_cases(num_cases=6, factors=factors, seed=0)
for c in cases[:3]:
    print(f'[{c.id}] expect={c.expected_behavior!r:20s} factors={c.factors}')
    print('      message:', c.user_message)

A designed suite is what turns a demonstration into evidence: it covers the hard combinations by construction and injects the failures an adversary would, so a passing run says something about cases no one wrote by hand. Chapter~12 assembles the governed harness these cases run against, and Chapter~16 runs the full suite and reports coverage.